In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Removing Duplicates**

In [0]:
df = spark.sql("SELECT * FROM `ete-catalog`.silver.customers")

In [0]:
df.limit(10).display()

# **Filtering Old and New Records**

**Using Table for filering**

In [0]:
flag = dbutils.widgets.get('load_flag')

if flag == 0 :
    df_old = spark.sql(''' SELECT DimCustomerKey, customer_id, create_date, update_date
                       FROM `ete-catalog`.gold.DimCustomers
                       ''')
else :
    df_old = spark.sql(''' SELECT 0 as DimCustomerKey, 0 as customer_id, 0 as create_date, 0 as update_date
                       FROM `ete-catalog`.silver.region
                       WHERE 1=0
                       ''')


In [0]:
df_old.display()

**Renaming These old columns with old as preceeding**

In [0]:
df_old = df_old.withColumnsRenamed({"DimCustomerKey" : "old_DimCustomerKey",
                           "customer_id" : "old_customer_id",
                           'create_date' : 'old_create_date',
                           "update_date" : "old_update_date"})

**Joining both dfs**

In [0]:
df_join = df.join(df_old,df['customer_id'] == df_old['old_customer_id'], 'left')

display(df_join.limit(12))

**Seperating old and new records**

In [0]:
df_new = df_join.filter(col('old_DimCustomerKey').isNull())

df_old = df_join.filter(col('old_DimCustomerKey').isNotNull())

In [0]:
df_new.display()